# BLOQUE 01 – PERFIL DEMOGRÁFICO - CONDICIONANTES
## Objetivo
Construir capa consolidada demográfica normalizada por unidad espacial.
## Inputs
- manzanas.gpkg
- censo_2020.csv
- conapo.xlsx
## Procesos
- Limpieza
- Homologación de CRS
- Join por CVEGEO

## Output
-1_D2C1V1._output_perfil_demografico.gpkg

In [ ]:
import pandas as pd
import geopandas as gpd

In [ ]:
# 1. RUTAS
ruta_censo = r"C:\Users\denih\OneDrive\Escritorio\DAL_LAB\data_row\censo\ageb_mza_urbana_14_cpv2020\conjunto_de_datos\conjunto_de_datos_ageb_urbana_14_cpv2020.csv"
ruta_gpkg = r"C:\Users\denih\OneDrive\Escritorio\DAL_LAB\data_row\manzanas.gpkg"
ruta_conapo = r"C:\Users\denih\OneDrive\Escritorio\DAL_LAB\data_row\conapo.xlsx"

In [17]:
# 2. CARGAR CONAPO CON LIMPIEZA DE ENCABEZADOS
# Leemos el excel sin encabezados primero para encontrar la fila correcta
df_temp = pd.read_excel(ruta_conapo, dtype=str, header=None)

# Buscamos en qué fila está la palabra "analfabeta" para que esa sea nuestra fila de nombres
fila_encabezado = df_temp[df_temp.apply(lambda x: x.astype(str).str.contains('analfabeta').any(), axis=1)].index[0]

# Volvemos a cargar usando esa fila como el header real
conapo = pd.read_excel(ruta_conapo, dtype=str, header=fila_encabezado)

# Limpiamos nombres de columnas (quitar espacios locos o saltos de línea)
conapo.columns = conapo.columns.str.strip().str.replace('\n', ' ')

# --- IDENTIFICAR COLUMNAS ---
col_clave_sucia = conapo.columns[7]
# Buscamos el nombre que contenga "analfabeta" por si cambia una letra
col_analfabeta = [c for c in conapo.columns if 'analfabeta' in c.lower()][0]
col_rezago = conapo.columns[27] # El 'Unnamed: 27' ahora debería tener su nombre real o posición

# Homologación
conapo['CVE_AGEB_FULL'] = conapo[col_clave_sucia].str.replace(r'\.0$', '', regex=True).str.zfill(13)
conapo_clean = conapo[['CVE_AGEB_FULL', col_analfabeta, col_rezago]].copy()

# 3. LEER Y FILTRAR CENSO
df_censo = pd.read_csv(ruta_censo, dtype=str, encoding='latin-1')
columnas_censo = ['POBTOT', 'POBFEM', 'POBMAS', 'P_0A2', 'P_0A2_F', 'P_0A2_M', 'P_3YMAS', 'P_3YMAS_F', 'P_3YMAS_M', 'P_5YMAS', 'P_5YMAS_F', 'P_5YMAS_M', 'P_12YMAS', 'P_12YMAS_F', 'P_12YMAS_M', 'P_15YMAS', 'P_15YMAS_F', 'P_15YMAS_M', 'P_18YMAS', 'P_18YMAS_F', 'P_18YMAS_M', 'P_3A5', 'P_3A5_F', 'P_3A5_M', 'P_6A11', 'P_6A11_F', 'P_6A11_M', 'P_8A14', 'P_8A14_F', 'P_8A14_M', 'P_12A14', 'P_12A14_F', 'P_12A14_M', 'P_15A17', 'P_15A17_F', 'P_15A17_M', 'P_18A24', 'P_18A24_F', 'P_18A24_M', 'P_15A49_F', 'P_60YMAS', 'P_60YMAS_F', 'P_60YMAS_M', 'POB0_14', 'POB15_64', 'POB65_MAS']

municipios_amg = ['039', '120', '098', '101', '097', '070', '044', '051', '124']
df_censo['MUN'] = df_censo['MUN'].str.zfill(3)
df_amg_censo = df_censo[df_censo['MUN'].isin(municipios_amg)].copy()

df_amg_censo['CVEGEO'] = (df_amg_censo['ENTIDAD'].str.zfill(2) + df_amg_censo['MUN'].str.zfill(3) + 
                          df_amg_censo['LOC'].str.zfill(4) + df_amg_censo['AGEB'].str.zfill(4) + 
                          df_amg_censo['MZA'].str.zfill(3))

# 4. CARGAR MANZANAS Y UNIR
manzanas_gpkg = gpd.read_file(ruta_gpkg)
manzanas_gpkg['CVEGEO'] = manzanas_gpkg['CVEGEO'].astype(str).str.strip()
manzanas_amg = manzanas_gpkg[manzanas_gpkg['CVEGEO'].str[2:5].isin(municipios_amg)].copy()
manzanas_amg['CVE_AGEB_FULL'] = manzanas_amg['CVEGEO'].str[:13]

# UNIONES
D2C1V1 = manzanas_amg.merge(df_amg_censo[['CVEGEO'] + columnas_censo], on='CVEGEO', how='left')
D2C1V1 = D2C1V1.merge(conapo_clean, on='CVE_AGEB_FULL', how='left')

# 5. GUARDAR
D2C1V1.to_file("D2C1V1.gpkg", driver="GPKG")

print(f"Éxito. Columna analfabeta encontrada como: '{col_analfabeta}'")

Éxito. Columna analfabeta encontrada como: 'Población de 15 años o más analfabeta'
